# **wiki2shards**

## Phase 0: Workbench (M0)

### Step 0.1: Create the skeleton and pin Python

```py
uv python pin 3.12.14
```
- The command `uv python pin 3.12.14` sets Python version 3.12.14 as the active Python version for your current working directory and project.
- Creates/Updates `.python-version`: It writes `3.12.14` to a `.python-version` file in the current directory.
- According to Charlie Marsh (creator of uv and founder of Astral), uv officially stands for "Ultraviolet".

```sh
mkdir -p src/wiki2shards/{io,acquire,extract,clean,tokenize,pack,loader,pipeline}
```
- `-p` stands for `parent` used in parent directory

```sh
touch src/wiki2shards/{io,acquire,extract,clean,tokenize,pack,loader,pipeline}/__init__.py
```

- creates a `__init__.py` inside every single directory within the curly brackets

```sh
echo "# wiki2shards" > README.md
```
- writes `# wiki2shards` into `README.md`

### Step 0.2: pyproject.toml

`pyproject.toml`
- `pyproject.toml` is the modern standard configuration file for Python projects. It centralizes project metadata, dependencies, and configuration for various development tools into a single file, replacing the older, fragmented ecosystem of setup.py, setup.cfg, and requirements.txt.

**`[project]`**
Declares the start of the core, standardized project metadata block (defined by PEP 621).

* **`name = "wiki2shards"`**
Sets the package's name. This is what you would use to pip install it.
* **`version = "0.1.0"`**
Sets the current version of the project.
* **`description = "Wikipedia dump -> checksummed token shards, plus a deterministic, resumable, mixture-aware loader"`**
Provides a short summary of the project's purpose.
* **`readme = "README.md"`**
Specifies that the long description (usually displayed on PyPI) should be pulled from the `README.md` file.
* **`requires-python = ">=3.12,<3.13"`**
Restricts this package to only run on Python 3.12.x. It will fail to install on Python 3.11 or Python 3.13.
* **`dependencies = [`**
Starts the list of core runtime dependencies required for the package to function.
* **`"numpy>=2.0",`**
Requires NumPy version 2.0 or higher.
* **`"mwparserfromhell==0.7.2",  # output bytes depend on it (wikitext goldens)`**
Pins this Wikipedia parsing library to exactly version 0.7.2. The comment explains this strict pinning ensures the processed output remains identical across builds.
* **`"tiktoken>=0.9",`**
Requires the `tiktoken` (OpenAI tokenizer) library version 0.9 or higher.
* **`"tokenizers==0.23.3",  # output bytes depend on it (fixture tokenizer)`**
Pins Hugging Face `tokenizers` to exactly 0.23.3 to guarantee deterministic tokenization.
* **`"pydantic>=2.7",`**
Requires Pydantic version 2.7 or higher for data validation.
* **`"typer>=0.12",`**
Requires Typer version 0.12 or higher to build the command-line interface.
* **`"pyyaml>=6.0",`**
Requires PyYAML version 6.0 or higher to parse YAML files.
* **`]`**
Closes the core dependencies list.

**`[project.scripts]`**
Defines console commands (CLI entry points) that get automatically installed with the package.

* **`wiki2shards = "wiki2shards.cli:main"`**
Creates a terminal command named `wiki2shards` that, when run, executes the `main` function inside the `wiki2shards.cli` Python module.

**`[dependency-groups]`**
Defines standard dependency groups (PEP 735) for different environments, natively supported by tools like `uv`.

* **`dev = [`**
Starts a group named `dev` containing dependencies only needed for developing the package, not running it.
* **`"pytest>=8",`**
Requires the Pytest testing framework version 8 or higher.
* **`"hypothesis>=6.100",`**
Requires Hypothesis v6.100+ for property-based testing.
* **`"pytest-timeout>=2.3",`**
Requires the pytest-timeout plugin v2.3+ to forcibly stop tests that hang.
* **`"ruff>=0.6",`**
Requires the Ruff linter/formatter version 0.6 or higher.
* **`"mypy>=1.11",`**
Requires the Mypy static type checker version 1.11 or higher.
* **`"types-PyYAML>=6.0",`**
Installs type hints for PyYAML so Mypy can properly type-check YAML-related code.
* **`]`**
Closes the dev dependencies list.

**`[build-system]`**
Defines the tools pip needs to build the package from source into a wheel (PEP 518).

* **`requires = ["hatchling"]`**
Specifies that the `hatchling` package must be downloaded before the build can start.
* **`build-backend = "hatchling.build"`**
Tells the build frontend (like pip) to use Hatchling's build system to package the code.

**`[tool.hatch.build.targets.wheel]`**
Configuration specific to the Hatchling build backend regarding how to build the `.whl` (wheel) file.

* **`packages = ["src/wiki2shards"]`**
Tells Hatchling that the actual Python code to be packaged lives in the `src/wiki2shards` directory (a standard `src` layout).

**`[tool.pytest.ini_options]`**
Configuration block for the Pytest testing framework.

* **`testpaths = ["tests"]`**
Tells Pytest to only look for tests inside the `tests` directory.
* **`pythonpath = ["tests"]`**
Adds the `tests` directory to the Python path during a test run, allowing test files to import helper functions from other test files.
* **`addopts = "-p no:cacheprovider --strict-markers"`**
Default command-line flags applied to every pytest run: disables test result caching (`-p no:cacheprovider`) and throws an error if an unregistered test marker is used (`--strict-markers`).
* **`timeout = 120`**
Instructs `pytest-timeout` to automatically fail any single test that takes longer than 120 seconds.
* **`markers = [`**
Starts a list to register custom test markers (decorators like `@pytest.mark.slow`).
* **`"slow: subprocess builds, crash matrix, SIGKILL resume (excluded from the fast suite)",`**
Registers the `slow` marker and provides a description of what it's used for.
* **`"bench: benchmarks, never gating",`**
Registers the `bench` marker used for benchmarking tests.
* **`]`**
Closes the markers list.

**`[tool.ruff]`**
Configuration block for the Ruff linter and formatter.

* **`line-length = 100`**
Sets the maximum allowed characters per line to 100 (overriding the default 88).
* **`target-version = "py312"`**
Tells Ruff to assume the code is running on Python 3.12, allowing it to apply modern formatting and linting rules specific to that version.

**`[tool.ruff.lint]`**
Specific configuration for Ruff's linting features.

* **`select = ["E", "F", "W", "I", "B", "UP", "SIM", "RUF"]`**
Enables specific families of linting rules: `E` and `W` (pycodestyle errors/warnings), `F` (Pyflakes), `I` (isort for imports), `B` (flake8-bugbear for common bugs), `UP` (pyupgrade for modern syntax), `SIM` (flake8-simplify), and `RUF` (Ruff's own rules).

**`[tool.mypy]`**
Configuration block for the Mypy static type checker.

* **`strict = true`**
Turns on Mypy's most restrictive mode, requiring type hints for everything and prohibiting dynamic typing loopholes.
* **`python_version = "3.12"`**
Tells Mypy to type-check against Python 3.12 rules.
* **`files = ["src"]`**
Instructs Mypy to only check the code inside the `src` directory (ignoring tests or scripts).

**`[[tool.mypy.overrides]]`**
Starts an override block (denoted by double brackets `[[ ]]` representing an array of tables in TOML) to apply specific Mypy rules to specific libraries.

* **`module = ["mwparserfromhell", "mwparserfromhell.*"]`**
Targets the `mwparserfromhell` library and all of its submodules for this override.
* **`ignore_missing_imports = true`**
Tells Mypy to suppress errors about this library missing type hints (since it does not provide them natively and no stub package exists).

### Step 0.3: Package marker, then install

`src/wiki2shards/__init__.py`

### Step 0.4: Crash-safe writes

`src/wiki2shards/io/atomic.py`

```python
"""Crash-safe writes: tmp -> fsync -> os.replace -> fsync(dir).

A reader can never observe a partially written file under its final name: the bytes land in a
hidden tmp file next to it, and only a complete, fsynced file is renamed into place. os.replace
is atomic on POSIX within one filesystem, which is why the tmp file lives in the same directory.
"""

```

* **Lines 1-6:** A module-level docstring summarizing the entire script's purpose. It explains the core mechanism of atomic file writes: data is written to a temporary file, synced to disk, and then atomically renamed over the original file so no other program ever sees a partially written state.

```python
from __future__ import annotations

```

* **Line 8:** Instructs Python to delay evaluating type hints. This allows you to use modern type hint syntax (like `Path | str`) even in older Python versions, and prevents circular import issues with typing.

```python
import os
from collections.abc import Iterator
from contextlib import contextmanager
from pathlib import Path
from typing import BinaryIO

```

* **Line 10:** Imports the standard `os` module, providing low-level operating system interfaces (like `fsync`, `replace`, `open`, `close`).
* **Line 11:** Imports `Iterator` for type hinting the return type of the context manager.
* **Line 12:** Imports the `contextmanager` decorator, which allows writing a function that behaves like a class with `__enter__` and `__exit__` methods for use in a `with` statement.
* **Line 13:** Imports `Path` from `pathlib` for modern, object-oriented filesystem path manipulation.
* **Line 14:** Imports `BinaryIO` for type hinting a file object opened in binary mode.

```python
TMP_SUFFIX = ".tmp"

```

* **Line 16:** Defines a constant string `.tmp` that will be appended to the end of temporary files.

```python
def tmp_name(final: Path) -> Path:
    """Hidden sibling used while `final` is being written."""
    return final.with_name(f".{final.name}{TMP_SUFFIX}")

```

* **Line 19:** Defines a function `tmp_name` that takes a `Path` object (the target file) and returns a new `Path` object.
* **Line 20:** A docstring explaining the function's purpose.
* **Line 21:** Generates the temporary file path. It takes the original filename (`final.name`), prepends a dot `.` to make it a hidden file on UNIX/Linux systems, and appends the `.tmp` suffix. It replaces the original filename in the original directory (`final.with_name(...)`).

```python
def fsync_dir(path: Path) -> None:
    """Persist a directory entry (the rename) itself."""
    fd = os.open(path, os.O_RDONLY)
    try:
        os.fsync(fd)
    finally:
        os.close(fd)

```

* **Line 24:** Defines a function `fsync_dir` that takes a directory path and returns nothing (`None`).
* **Line 25:** A docstring explaining that this persists the directory metadata (necessary to guarantee a file rename survives a power loss).
* **Line 26:** Uses the low-level `os.open` to open the directory itself in read-only mode (`os.O_RDONLY`), returning an integer file descriptor (`fd`).
* **Line 27:** Starts a `try` block to ensure cleanup happens even if an error occurs.
* **Line 28:** Calls `os.fsync()` on the directory's file descriptor, forcing the operating system to flush the directory's metadata changes to physical storage.
* **Line 29:** Starts the `finally` block, which executes regardless of whether the `try` block succeeded or threw an exception.
* **Line 30:** Closes the low-level file descriptor to prevent resource leaks.

```python
@contextmanager
def atomic_open(final: Path | str) -> Iterator[BinaryIO]:
    """Open a binary writer whose content appears at `final` only if the block succeeds."""

```

* **Line 33:** Applies the `@contextmanager` decorator, converting the generator function below into a context manager that can be used with a `with` statement.
* **Line 34:** Defines the `atomic_open` function, which accepts either a `Path` or a string, and yields a binary file object (`BinaryIO`).
* **Line 35:** A docstring explaining that the file will only appear at its final destination if the code block inside the `with` statement completes without errors.

```python
    final = Path(final)
    tmp = tmp_name(final)
    f = open(tmp, "wb")  # noqa: SIM115 - closed explicitly on both paths below

```

* **Line 36:** Normalizes the input `final` by ensuring it is a `Path` object (in case a string was passed).
* **Line 37:** Calls `tmp_name()` to determine the hidden temporary file path.
* **Line 38:** Opens the temporary file for writing in binary mode (`"wb"`). The `# noqa: SIM115` comment tells a specific linter (flake8-simplify) to ignore the warning about opening a file without using a `with` statement, since this function manually manages the closing.

```python
    try:
        yield f
        f.flush()
        os.fsync(f.fileno())
        f.close()
        os.replace(tmp, final)
        fsync_dir(final.parent)

```

* **Line 39:** Starts the `try` block for the "success" path.
* **Line 40:** `yield f` hands the open temporary file object back to the caller's `with` block. Execution pauses here until the user's `with` block finishes.
* **Line 41:** Once the user's block finishes successfully, execution resumes. `f.flush()` forces Python's internal memory buffers to write to the OS.
* **Line 42:** `os.fsync(f.fileno())` tells the OS to physically write its own disk buffers to the hardware storage, guaranteeing the data is saved.
* **Line 43:** Closes the temporary file. It is now safely written to disk, but still has the `.tmp` name.
* **Line 44:** `os.replace` atomically renames the temporary file to the final filename. If a file already exists at `final`, it is silently and atomically overwritten.
* **Line 45:** Calls the previously defined `fsync_dir` on the parent directory (`final.parent`) to guarantee that the OS persists the file rename to disk.

```python
    except BaseException:
        f.close()
        tmp.unlink(missing_ok=True)
        raise

```

* **Line 46:** The `except BaseException` block catches *any* exception that occurred while the user was interacting with the yielded file in the `with` block, including system-exiting exceptions like `KeyboardInterrupt`.
* **Line 47:** Closes the file object to release the resource.
* **Line 48:** Deletes the temporary file (`unlink`). `missing_ok=True` prevents a `FileNotFoundError` if the file somehow doesn't exist.
* **Line 49:** `raise` re-throws the original exception so the calling program knows an error occurred and can handle it or crash normally.

```python
def atomic_write_bytes(final: Path | str, data: bytes) -> None:
    with atomic_open(final) as f:
        f.write(data)

```

* **Line 52:** Defines a helper function that takes a target path and a payload of raw bytes, returning nothing.
* **Line 53:** Uses the `atomic_open` context manager defined above to safely open the destination file.
* **Line 54:** Writes the entire `data` byte payload into the yielded file object. Once this block ends, `atomic_open` handles flushing, syncing, and renaming.

```python
def commit(src: Path, final: Path) -> None:
    """Move an already-complete file (written elsewhere in the same filesystem) into place."""
    with open(src, "rb") as f:
        os.fsync(f.fileno())
    os.replace(src, final)
    fsync_dir(final.parent)

```

* **Line 57:** Defines a `commit` function intended to take an existing file (`src`) and safely finalize it as `final`.
* **Line 58:** A docstring explaining its use case.
* **Line 59:** Opens the source file in read-binary mode using a standard `with` context manager.
* **Line 60:** Calls `os.fsync()` on the source file to ensure any pending writes made by whoever created the file are physically on disk before moving it.
* **Line 61:** The `with` block ends, automatically closing the source file.
* **Line 62:** Atomically renames the source file to its final destination.
* **Line 63:** Syncs the destination's parent directory to ensure the rename operation is permanently recorded on disk.

### Step 0.5: Hashing, canonical JSON and document identity

`src/wiki2shards/io/hashing.py`

> File Header & Setup

* `"""Stable hashing, canonical JSON and document identity.` — The start of a multi-line comment describing what this file does.
* `Nothing here uses the built-in hash(): str hashing is salted per process (PYTHONHASHSEED), so any` — Explains that Python's default tool for mixing data (`hash()`) changes every time you restart the program.
* `value derived from it would differ between runs. Everything persisted goes through sha256.` — Because default hashes change, this code uses `sha256` (a standard mathematical formula) to ensure digital fingerprints stay exactly the same forever.
* `"""` — The end of the multi-line comment.
* `from __future__ import annotations` — Allows the code to use modern Python type hints (labels telling you what type of data a variable is) even on older Python versions.
* `import hashlib` — Brings in Python's built-in tool for creating secure digital fingerprints (hashes).
* `import json` — Brings in the tool to read and write JSON, a very common text format used to store and send data.
* `from pathlib import Path` — Brings in a tool that makes it easy to work with computer file and folder paths.
* `from typing import Any` — Brings in a special label (`Any`) used to tell the code "this variable can hold absolutely any type of data."
* `_CHUNK = 1 << 20` — Creates a constant variable set to 1,048,576 (which is exactly 1 Megabyte). This is used to read large files in small pieces so they don't crash the computer's memory.

> Hashing Functions (Making Digital Fingerprints)

* `def sha256_hex(data: bytes) -> str:` — Defines a function named `sha256_hex` that takes raw computer bytes and returns a standard text string.
* `return hashlib.sha256(data).hexdigest()` — Creates a SHA-256 fingerprint of the bytes and hands it back as readable text (hexadecimal).
* `def sha256_file(path: Path | str) -> str:` — Defines a function that takes a file path (as a Path object or text string) and returns a text string.
* `h = hashlib.sha256()` — Starts an empty SHA-256 fingerprint calculator.
* `with open(path, "rb") as f:` — Safely opens the file in "read binary" (`rb`) mode so it can read the raw 1s and 0s.
* `while chunk := f.read(_CHUNK):` — A loop that reads the file one 1-Megabyte chunk at a time until the file is empty.
* `h.update(chunk)` — Adds that 1-Megabyte chunk into the fingerprint calculator.
* `return h.hexdigest()` — Once the whole file is read, hands back the final fingerprint as text.
* `def sha1_file(path: Path | str) -> str:` — Defines a function exactly like the one above, but for an older, faster fingerprint format called SHA-1.
* `h = hashlib.sha1()` — Starts an empty SHA-1 fingerprint calculator.
* `with open(path, "rb") as f:` — Safely opens the file in "read binary" mode.
* `while chunk := f.read(_CHUNK):` — Reads the file one 1-Megabyte chunk at a time.
* `h.update(chunk)` — Adds the chunk into the SHA-1 calculator.
* `return h.hexdigest()` — Hands back the final SHA-1 fingerprint as text.

> Strict Data Checking

* `def _check(obj: Any, where: str = "$") -> None:` — Defines a hidden function that inspects data to ensure it is perfectly safe for strict JSON. The `where` variable tracks where we are in the data structure (starting at `$`, the root).
* `"""Reject anything whose JSON encoding is not unique: floats, NaN, non-string keys."""` — A comment explaining that decimal numbers and weird dictionary keys are banned because different computers format them differently.
* `if isinstance(obj, bool) or obj is None or isinstance(obj, (int, str)):` — Checks if the data is a safe, simple type: True/False, empty (`None`), a whole number, or text.
* `return` — If it is safe, stop checking and approve it (do nothing).
* `if isinstance(obj, float):` — Checks if the data is a decimal number (a float).
* `raise TypeError(f"float at {where}: canonical JSON is integers only")` — If it is a decimal, crash the program and print an error showing exactly where the decimal was found.
* `if isinstance(obj, dict):` — Checks if the data is a dictionary (a collection of key-value pairs).
* `for k, v in obj.items():` — Loops through every single key (`k`) and value (`v`) inside the dictionary.
* `if not isinstance(k, str):` — Checks if the dictionary key is anything other than text.
* `raise TypeError(f"non-string key {k!r} at {where}")` — If the key isn't text, crash the program with an error.
* `_check(v, f"{where}.{k}")` — Takes the dictionary value and sends it back through this exact same `_check` function to inspect it. This ensures nested data is checked all the way down.
* `return` — Once the whole dictionary is checked, stop.
* `if isinstance(obj, (list, tuple)):` — Checks if the data is a list or tuple (an ordered collection of items).
* `for i, v in enumerate(obj):` — Loops through the list, keeping track of the item's position number (`i`) and the item itself (`v`).
* `_check(v, f"{where}[{i}]")` — Sends each list item back through the `_check` function to inspect it, recording its position.
* `return` — Once the whole list is checked, stop.
* `raise TypeError(f"unsupported type {type(obj).__name__} at {where}")` — If the data isn't any of the allowed types above, crash the program and say the data type is unsupported.

> JSON Formatting (Canonical JSON)

* `def canonical_json(obj: Any) -> bytes:` — Defines a function that turns data into a highly strict, squished-down JSON byte format.
* `"""Compact form: stage keys, JSONL lines, journals."""` — A comment saying this squished format is meant for tight storage and logs.
* `_check(obj)` — Runs the data through our `_check` function above to make sure it contains no banned elements.
* `text = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"))` — Converts the data into text JSON. It forces the keys into alphabetical order, allows foreign language characters, and removes all extra spaces.
* `return text.encode("utf-8")` — Converts that strict JSON text into raw computer bytes using the standard UTF-8 format.
* `def canonical_json_pretty(obj: Any) -> bytes:` — Defines a function that turns data into strict JSON, but formatted beautifully for human eyes.
* `"""Manifest form: sorted keys, indent 2, UTF-8, trailing newline."""` — A comment explaining this format is spaced out and easy to read.
* `_check(obj)` — Checks the data for banned elements again.
* `text = json.dumps(obj, sort_keys=True, ensure_ascii=False, indent=2, separators=(",", ": "))` — Converts the data into text JSON, sorts the keys, adds a 2-space indentation for neatness, and puts spaces after colons.
* `return (text + "\n").encode("utf-8")` — Adds an invisible "Enter" (new line) at the very end of the text, then converts it to bytes.

> Document Identity (Unique IDs)

* `def doc_uid(wiki: str, page_id: int) -> int:` — Defines a function that creates a unique ID number by combining a wiki name and a page number.
* `"""Little-endian u64 of the first 8 bytes of sha256(f"{wiki}:{page_id}")."""` — A comment explaining the math used to generate the ID.
* `digest = hashlib.sha256(f"{wiki}:{page_id}".encode()).digest()` — Mashes the wiki name and page ID together into text (like "enwiki:123"), converts it to bytes, and generates a raw SHA-256 fingerprint.
* `return int.from_bytes(digest[:8], "little")` — Chops off just the first 8 bytes of that fingerprint and converts them into a massive whole number.
* `def uid_hex(uid: int) -> str:` — Defines a function that takes that massive whole number ID and turns it into text.
* `"""JSON form of a doc_uid: hex of the integer, so string order equals numeric order."""` — Explains this makes the ID safe to save in JSON while making sure alphabetical order matches numerical order.
* `return f"{uid:016x}"` — Converts the number into a 16-character text string using letters and numbers, adding zeros to the front if it is too short.
* `def parse_uid_hex(text: str) -> int:` — Defines a function that takes the 16-character text ID and turns it back into a number.
* `if len(text) != 16:` — Checks if the text is exactly 16 characters long.
* `raise ValueError(f"doc_uid must be 16 hex digits, got {text!r}")` — If it isn't exactly 16 characters, crash the program with an error.
* `return int(text, 16)` — Converts the 16-character text string back into a standard whole number.
* `def text_sha256(text: str) -> str:` — Defines a simple function to quickly get the fingerprint of plain text.
* `return sha256_hex(text.encode("utf-8"))` — Converts the text into raw bytes and passes it to our very first function (`sha256_hex`) to get the fingerprint.

### Step 0.6: Fault injection

`src/wiki2shards/pipeline/faults.py`

| Code Line | Plain English Explanation |
| --- | --- |
| `"""Fault injection: W2S_FAULT="<stage>:<point>:<n>" kills the coordinator at the n-th hit.` | Opens a multi-line comment describing what this code does: it's a tool to intentionally crash a program by setting a specific rule. |
| `os._exit skips finally blocks, atexit handlers and buffered flushes, exactly like SIGKILL, so the` | Continues the comment, explaining that this tool forces a brutal, instant crash without giving the program any time to clean up its files or memory. |
| `crash tests see the same on-disk state a real kill would leave behind.` | Concludes the comment, noting that this helps developers test how the system reacts to sudden power cuts or forced closures. |
| `"""` | Closes the multi-line comment block. |
| `from __future__ import annotations` | A special instruction that helps developers write clearer, more modern code labels (type hints) in older versions of Python. |
| `import os` | Brings in Python's built-in operating system tools, which are needed to read computer settings and force the program to quit. |
| `FAULT_EXIT_CODE = 137` | Sets a specific error number (137) that the program will announce when it intentionally crashes itself. |
| `_hits: dict[tuple[str, str], int] = {}` | Creates an empty digital scoreboard (a dictionary) to keep a running tally of how many times different parts of the code are reached. |
| `def hit(stage: str, point: str) -> None:` | Creates a reusable block of code (a function) named `hit`. You give it a `stage` (a phase of the program) and a `point` (a specific spot). |
| `    spec = os.environ.get("W2S_FAULT")` | Looks at your computer's hidden settings (environment variables) to see if you gave it a crash instruction named "W2S_FAULT". |
| `    if not spec:` | Checks if that crash instruction is missing or empty. |
| `        return` | If there is no crash instruction, exit this block of code immediately and let the program continue running normally. |
| `    want_stage, want_point, n = spec.split(":")` | If there *is* an instruction, chop it into three pieces separated by colons: the exact stage, the exact point, and the number of times to hit it. |
| `    if (stage, point) != (want_stage, want_point):` | Checks if the program's current location matches the exact location where you told it to crash. |
| `        return` | If the locations don't match, exit this block of code and let the program continue normally. |
| `    count = _hits.get((stage, point), 0) + 1` | Checks the scoreboard to see how many times the program has visited this exact spot before, and adds 1 to that number. |
| `    _hits[(stage, point)] = count` | Updates the scoreboard with the new, higher count. |
| `    if count == int(n):` | Checks if the number of times we've visited this spot exactly matches the target crash number you set. |
| `        os._exit(FAULT_EXIT_CODE)` | If the numbers match, instantly and violently kill the program, spitting out the error number 137. |

### Step 0.7: The fixture dump generator

`scripts/make_fixture_dump.py`

Here is a plain-English breakdown of what this script does, section by section.

This script generates **fake Wikipedia database dumps** for testing software. Instead of downloading gigabytes of real Wikipedia data, developers can run this script to generate a small, predictable, "dummy" version of Wikipedia that always looks exactly the same every time it is generated.

> 1. Introduction and Setup

* **The Docstring (`""" ... """`)**: Explains what the script does. It creates files that perfectly mimic the structure of real Wikimedia dump files (XML compressed with bz2). It emphasizes that it uses no random number generators—everything is strictly calculated so the output never changes between computers.
* **Imports**: Loads built-in Python tools needed for the job: `argparse` (to read command-line commands), `bz2` (to compress files), `hashlib` (for generating predictable numbers/checksums), `json` (for saving test answers), `Path` (for handling file folders), and `escape/quoteattr` (to safely format text into XML).

> 2. The Raw Ingredients (Constants)

* `PAGES_PER_STREAM = 100`: Groups 100 wiki pages into a single compressed block, just like real Wikipedia dumps.
* `BASE_DATE`: A fake date used for the file names.
* `WORDS`: A giant list of simple English words. The script uses these to generate fake sentences and page titles.
* `COUNTS`: A dictionary deciding exactly how many pages of specific "weird" types to create (e.g., 20 redirects, 10 talk pages, 1 emoji page).
* `TOTAL_PAGES = 300`: The script will generate exactly 300 pages in total.

> 3. Generating Predictable "Random" Text

Instead of using Python's `random` module (which can act differently on different machines), the script uses math to pick words.

* **`_h` function**: Takes any input (like a page name), hashes it using SHA-256, and turns it into a permanent, unique number.
* **`words` function**: Uses that unique number to pick words from the `WORDS` list over and over to form a list of fake words.
* **`prose` function**: Takes those words, capitalizes the first letter, adds periods, and groups them into 5-sentence paragraphs to look like real article text.
* **`article_wikitext` function**: Glues the fake paragraphs together with real MediaWiki formatting (like `[[links]]`, `{{infoboxes}}`, `<ref>` tags, and tables) so the testing software has complex formatting to practice reading.

> 4. Building the Pages

* **`Page` class**: A simple blueprint to hold a page's data (its title, category, namespace ID, and text).
* **`build_pages` function**:
* Figures out how many standard "article" pages to make to hit exactly 300 pages.
* Loops through every category (redirects, empty pages, short pages, foreign languages, etc.) and generates the text for them using the functions above.
* Sorts them using the `_h` math function so they are thoroughly—but predictably—mixed up.
* If `variant 2` is requested (simulating a wiki update that happens later in time), it deletes 5 pages, edits a few, and adds 5 new ones.

> 5. Keeping IDs Stable

* **`page_ids` function**: Real Wikipedia pages have ID numbers that never change, even if other pages are deleted. This function calculates permanent ID numbers for every generated page.

> 6. Formatting as XML

* **`page_xml` function**: Takes a `Page` object and wraps it in MediaWiki XML tags (like `<page>`, `<title>`, `<text>`).
* **`header_xml` function**: Creates the standard XML header that sits at the very top of the dump file, declaring the site's name as "Fixturepedia".

> 7. Creating the "Answer Key"

* **`expected_answers` function**: Since this script generates test data, it also generates a "cheat sheet" (a JSON file). This file calculates exactly how many pages are valid, how many are empty, and how many are duplicates. The testing software will use this JSON file to grade itself and make sure it parsed the wiki dump correctly.

> 8. Compressing and Saving the Files

* **`make_dump` function**: This is where the actual files are built.
* It gathers all the pages and their IDs.
* It breaks the pages into chunks of 100.
* It converts them to XML, creates an index (so software knows exactly where in the file a specific page lives), and heavily compresses them into `.bz2` format.
* It creates the folder structure (`data/fixture_dump/fixturewiki/20260901/`).
* It writes the compressed XML file, the compressed index file, a `sha1sums.txt` file (to prove the files aren't corrupted), and the `fixture_expected.json` answer key to your hard drive.

> 9. The Execution Trigger

* **`main` function**: Sets up the command-line interface. When you type `python make_fixture_dump.py --out data`, this function reads your instructions and triggers `make_dump`.
* **`if __name__ == "__main__":`**: A standard Python rule that says, "If someone runs this script directly, start the `main()` function."

### Step 0.8: Shared test setup and the first tests

#### `tests/conftest.py`

`conftest.py` registers a deterministic Hypothesis profile and blocks network sockets. It also defines session-scoped fixtures that build the fixture dataset once per test run; those use helpers that arrive in later phases and are imported lazily.

```python
"""Shared test setup: Hypothesis profiles, a network guard, and session-scoped fixture builds."""

```

This is a comment at the very top of the file (a docstring) explaining what this file does: it sets up testing tools, blocks internet access during tests, and creates reusable test data.

```python
from __future__ import annotations

```

This allows Python to use modern type-hinting features (where you specify what type of data a variable is) without crashing on older Python versions.

```python
import atexit

```

Imports a built-in Python tool that lets you schedule tasks to run right before the program finishes and closes.

```python
import os

```

Imports a built-in tool to interact with your operating system, like reading environment variables or navigating folders.

```python
import shutil

```

Imports a built-in tool for managing files and folders, like deleting entire directories.

```python
import socket

```

Imports a built-in tool that Python uses to create network and internet connections.

```python
import tempfile

```

Imports a built-in tool used to create temporary files and folders that won't clutter your hard drive.

```python
from pathlib import Path

```

Imports a modern tool that makes it easier to work with file and folder paths across different operating systems (Windows, Mac, Linux).

```python
import pytest

```

Imports `pytest`, which is the main framework being used to write and run the tests.

```python
from hypothesis import HealthCheck, settings

```

Imports specific tools from `hypothesis`, a library that automatically generates test data to help find bugs.

```python
from hypothesis.configuration import set_hypothesis_home_dir

```

Imports a function to change the folder where `hypothesis` saves its background files and caches.

```python
REPO = Path(__file__).resolve().parents[1]

```

Calculates the exact folder path to the main project directory. It looks at the current file's path (`__file__`), gets the absolute path (`resolve`), and goes up two folder levels (`parents[1]`).

```python
os.chdir(REPO)  # configs use CWD-relative paths (tokenizer, language codes)

```

Changes the current working directory to the main project folder so that any files the tests need to read are found in the right place.

```python
# Hypothesis keeps caches (constants, unicode data) in ./.hypothesis even with database=None.
# Tests write only under temporary directories, so point its home at one for this session.

```

Comments explaining why the next few lines are necessary: to prevent `hypothesis` from leaving leftover cache files in the project folder.

```python
_HYPOTHESIS_HOME = tempfile.mkdtemp(prefix="w2s-hypothesis-")

```

Creates a new temporary folder on your computer starting with the name `w2s-hypothesis-` and saves its path to the variable `_HYPOTHESIS_HOME`.

```python
atexit.register(shutil.rmtree, _HYPOTHESIS_HOME, True)

```

Tells Python: "When the tests are completely finished, use `shutil.rmtree` to delete this temporary folder so it doesn't leave junk on the computer."

```python
set_hypothesis_home_dir(_HYPOTHESIS_HOME)

```

Tells `hypothesis` to use that temporary folder for its background files instead of the default project folder.

```python
# "ci" is deterministic and writes no example database (tests write only under tmp_path).

```

A comment explaining that the next block configures a testing profile for "CI" (Continuous Integration - the automated server that runs tests). It says it will be predictable and won't save files to the hard drive.

```python
settings.register_profile(

```

Starts the process of saving a specific configuration setup (a "profile") for `hypothesis`.

```python
    "ci",

```

Names this specific configuration profile "ci".

```python
    max_examples=60,

```

Tells `hypothesis` to generate and run a maximum of 60 random test cases per test.

```python
    deadline=None,

```

Turns off the time limit, meaning tests won't automatically fail if they run a little slowly.

```python
    derandomize=True,

```

Forces the randomly generated test data to be exactly the same every single time you run the tests, which makes debugging easier.

```python
    database=None,

```

Tells `hypothesis` not to save previously failed test cases to a database on the hard drive.

```python
    suppress_health_check=[HealthCheck.too_slow],

```

Tells `hypothesis` not to complain or throw a warning if generating the test data takes a long time.

```python
)

```

Closes the profile configuration block.

```python
settings.register_profile("explore", max_examples=500, deadline=None, database=None)

```

Creates a second configuration profile named "explore" that is more aggressive: it runs 500 examples instead of 60 to hunt for deeper bugs.

```python
settings.load_profile(os.environ.get("HYPOTHESIS_PROFILE", "ci"))

```

Activates a profile. It checks if your computer has an environment variable called `HYPOTHESIS_PROFILE` set (like "explore"). If not, it defaults to using the "ci" profile we just made.

```python
@pytest.fixture(autouse=True)

```

A special tag (decorator) that tells `pytest`: "The following function is a setup task, and you should run it automatically before every single test."

```python
def _no_network(monkeypatch: pytest.MonkeyPatch) -> None:

```

Defines a function named `_no_network` that takes a tool called `monkeypatch` (which lets you temporarily replace parts of Python). `-> None` means this function doesn't return a value.

```python
    """Tests never touch the network: refuse any internet socket connection."""

```

A comment explaining the function's purpose: to strictly block the tests from accessing the internet.

```python
    real_connect = socket.socket.connect

```

Saves a copy of Python's real, original function that connects to the internet, so it can still be used for local (non-internet) connections.

```python
    def guarded(self: socket.socket, address: object) -> None:

```

Defines a fake, custom connection function inside our setup function, which will act as a security guard.

```python
        if self.family in (socket.AF_INET, socket.AF_INET6):

```

Checks if the connection being attempted is an IPv4 or IPv6 internet connection.

```python
            raise RuntimeError(f"network access attempted in a test: {address!r}")

```

If it *is* an internet connection, this deliberately crashes the test with an error message showing what address it tried to connect to.

```python
        real_connect(self, address)  # type: ignore[arg-type]

```

If it is a safe local connection (like reading a local file socket), it goes ahead and uses the `real_connect` function we saved earlier.

```python
    monkeypatch.setattr(socket.socket, "connect", guarded)

```

Takes our fake `guarded` function and temporarily replaces Python's built-in internet connection tool with it for the duration of the tests.

```python
@pytest.fixture(scope="session")

```

A tag telling `pytest`: "The following setup function is expensive, so only run it once for the entire test session, and let all tests share the result."

```python
def dumps(tmp_path_factory: pytest.TempPathFactory) -> Path:

```

Defines a function named `dumps` that asks `pytest` for a tool to make temporary folders (`tmp_path_factory`). It promises to return a file `Path`.

```python
    """Fixture dumps: fixturewiki at two dates, plus a second wiki for two-source mixtures."""

```

A comment explaining that this function builds fake Wikipedia data dumps for testing purposes.

```python
    from helpers.build import make_dumps

```

Imports a custom helper function named `make_dumps` from another file in the project. (It imports it here instead of at the top of the file so it only loads if this function actually runs).

```python
    return make_dumps(tmp_path_factory.mktemp("dumps"))

```

Creates a temporary folder named "dumps", builds the fake data inside it using `make_dumps`, and hands that folder's path to any test that asks for it.

```python
@pytest.fixture(scope="session")

```

Another tag telling `pytest` to run the following setup function only once for the whole test session.

```python
def fixture_build(dumps: Path, tmp_path_factory: pytest.TempPathFactory):  # type: ignore[no-untyped-def]

```

Defines a function named `fixture_build` that requires the `dumps` folder we just made in the previous step, and the temporary folder tool.

```python
    from helpers.build import build_fixture

```

Imports another custom helper function called `build_fixture` from a local project file.

```python
    return build_fixture(tmp_path_factory.mktemp("build_a"), dumps)

```

Creates a new temporary folder named "build_a", processes the `dumps` data into a usable test environment, and returns it for tests to use.

```python
@pytest.fixture(scope="session")

```

Another tag telling `pytest` to run this setup function only once per session.

```python
def fixture_build_b(dumps: Path, tmp_path_factory: pytest.TempPathFactory):  # type: ignore[no-untyped-def]

```

Defines a function named `fixture_build_b` to create a second, slightly different test environment, also requiring the original `dumps` folder.

```python
    from helpers.build import build_fixture

```

Imports the `build_fixture` tool again.

```python
    return build_fixture(tmp_path_factory.mktemp("build_b"), dumps, wiki="fixture2wiki")

```

Creates a temporary folder named "build_b" and builds a second environment based on a different fake wiki named "fixture2wiki", making it available for tests that need two different setups at once.

#### `tests/unit/test_io_atomic.py`

This code tests a custom file-saving tool to ensure it saves files "atomically"—meaning the save is all-or-nothing. If a crash happens while saving, it guarantees you won't be left with a corrupted, half-written file.

Here is what every line does:

* `import subprocess`: Brings in a Python tool that allows this script to launch and run other programs in the background.
* `import sys`: Brings in a tool to interact with the Python system itself (used later to find the Python launcher).
* *(blank line)*
* `import pytest`: Brings in a popular testing framework used to verify that the code works as expected.
* *(blank line)*
* `from wiki2shards.io.atomic import atomic_open, atomic_write_bytes`: Loads two custom tools from the project's own code designed to safely write files without leaving them half-finished.
* *(blank line)*
* *(blank line)*
* `def test_atomic_write(tmp_path):`: Defines a new test function and gives it a temporary, disposable folder (`tmp_path`) to run tests inside safely.
* `    final = tmp_path / "out.bin"`: Creates a file path named "out.bin" inside that temporary folder.
* `    with pytest.raises(RuntimeError), atomic_open(final) as f:`: Tells the test to *expect* a `RuntimeError` crash to happen in the next block, and opens the file so we can start writing to it.
* `        f.write(b"partial")`: Writes the word "partial" into the file (the `b` means it is written as raw bytes, not standard text).
* `        raise RuntimeError("boom")`: Deliberately forces the program to crash with an error named "boom" right in the middle of writing.
* `    assert not final.exists()`: Checks to ensure the final "out.bin" file does not exist, proving the failed save didn't leave a half-written file behind.
* `    assert list(tmp_path.iterdir()) == []  # the tmp file is gone too`: Looks inside the temporary folder to guarantee no hidden leftover junk files were left behind either.
* *(blank line)*
* `    atomic_write_bytes(final, b"v1")`: Successfully writes the text "v1" into the file using the safe saving tool.
* `    with pytest.raises(RuntimeError), atomic_open(final) as f:`: Again, sets up a block that expects a crash, and opens the file to try and overwrite it.
* `        f.write(b"v2, half written")`: Starts writing new data over the old file.
* `        raise RuntimeError("boom")`: Deliberately forces another crash before the save can finish.
* `    assert final.read_bytes() == b"v1"  # the old content survives a failed rewrite`: Reads the file and verifies it still perfectly contains the original "v1" data, untouched by the failed overwrite attempt.
* *(blank line)*
* `    # A hard kill mid-write runs no cleanup at all; still nothing appears under the final name.`: A comment explaining that the next test will brutally kill the program without letting it clean up its mess.
* `    killed = tmp_path / "killed.bin"`: Creates a new file path named "killed.bin" in the temporary folder.
* `    code = (`: Begins a block of text that contains a miniature Python script.
* `        "import os, sys\nfrom wiki2shards.io.atomic import atomic_open\n"`: The mini-script loads necessary system tools and the custom safe-open tool.
* `        "with atomic_open(sys.argv[1]) as f:\n    f.write(b'x' * 4096)\n    f.flush()\n"`: The mini-script opens the file, writes 4,096 "x" characters, and forces the computer to physically push that data to the hard drive.
* `        "    os._exit(137)\n"`: The mini-script instantly and violently kills itself (using error code 137), completely bypassing any normal file-closing or cleanup steps.
* `    )`: Closes the block of text containing the mini-script.
* `    proc = subprocess.run([sys.executable, "-c", code, str(killed)])`: Uses the system to launch a completely separate background process to run that mini-script, pointing it at the "killed.bin" file path.
* `    assert proc.returncode == 137`: Verifies that the background process actually died violently with the 137 error code.
* `    assert not killed.exists()`: Checks to ensure that even though the program was killed brutally mid-save, the "killed.bin" file was never officially created, preventing data corruption.